In [3]:
import pandas as pd
import numpy as np
import json
import joblib
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import optuna
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, average_precision_score,
    roc_auc_score, confusion_matrix,
)

In [4]:
ABT_DIR = "../../data/Group2_ABT.csv"

df = pd.read_csv(ABT_DIR)
df.head()

C:\Users\Lenovo Thinkpad T460\AppData\Local\Temp\ipykernel_16744\2301934033.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Group2_ABT.csv')


,station,lat,datetime_utc,pm25 in µg/m^3,lon,city,province,population,area_in_sq.km,density_persons/sqkm
0,313_Cubao,14.6227,2025-09-30 07:00:00+00:00,19.652000,121.0528,Quezon,NCR,"14,001,751",619.54,"21,765"
1,313_Cubao,14.6227,2025-09-30 22:00:00+00:00,52.500000,121.0528,Quezon,NCR,"14,001,751",619.54,"21,765"
2,313_Cubao,14.6227,2025-09-30 23:00:00+00:00,41.965681,121.0528,Quezon,NCR,"14,001,751",619.54,"21,765"
3,313_Cubao,14.6227,2025-10-01 00:00:00+00:00,30.948667,121.0528,Quezon,NCR,"14,001,751",619.54,"21,765"
4,313_Cubao,14.6227,2025-10-01 01:00:00+00:00,30.113639,121.0528,Quezon,NCR,"14,001,751",619.54,"21,765"


### Dropping columns that won't needed

In [5]:
col_to_drop = ['lat', 'lon', 'province', 'population', 'area_in_sq.km']
df.drop(columns=col_to_drop, inplace=True)

### columns renaming

In [6]:
df = df.rename(columns={
    "pm25 in µg/m^3": "pm25_ug_m3",
    "density_persons/sqkm": "density",
})

## PARSE AND SORT

In [7]:
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True, errors="coerce")
df = df.dropna(subset=["datetime_utc", "pm25_ug_m3"])
df = df.sort_values(["station", "datetime_utc"])

## Per-station hourly resample (mean, not sum)
The data has no fixed interval, this code bins from 10am to 10:59am as 10am, and the rest is so on and so forth. Additionally, filled in the jumps of hours(e.g. from 5 jumped to 8 will now includ 6 and 7) but the station, density, and pm2.5 is NaN. The stations and city is retained, but the pm2.5 is averaged. 

reference: Robust prediction of hourly PM2.5 from meteorological data using LightGBM - PMC (https://pmc.ncbi.nlm.nih.gov/articles/PMC8566180/)

In [8]:
hourly = (df.set_index("datetime_utc")
            .groupby("station")
            .resample("1h", origin="epoch")
            .agg(pm25_ug_m3=("pm25_ug_m3", "mean"),
                 city=("city", "first"),
                 density=("density", "first"))
            .reset_index())

C:\Users\Lenovo Thinkpad T460\AppData\Local\Temp\ipykernel_16744\705913747.py:4: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .agg(pm25_ug_m3=("pm25_ug_m3", "mean"),


## Filling in the missing gaps

Since the jumped hours has been filled in but has missing values, we used linear interpolation and ffill() and bfill() for station and density as they do not change.

In [9]:
hourly = hourly.sort_values(["station", "datetime_utc"]).copy()

hourly["city"]    = hourly.groupby("station")["city"].ffill().bfill()
hourly["density"] = hourly.groupby("station")["density"].ffill().bfill()

# If a station has ALL city/density missing, groupby ffill can't help.
# Fallback: fill with the global mode / median so nothing is NaN.

# Convert density from string-with-commas to numeric
hourly["density"] = (
    hourly["density"]
        .astype(str)                     # ensure it's string (in case of mixed types)
        .str.replace(",", "", regex=False)  # remove thousands separators
        .str.strip()                     # remove stray whitespace
        .replace({"": None, "nan": None, "None": None, "N/A": None})
        .pipe(pd.to_numeric, errors="coerce")  # convert to float, bad → NaN
)

hourly["city"]    = hourly["city"].fillna(hourly["city"].mode().iloc[0])
hourly["density"] = hourly["density"].fillna(hourly["density"].median())

## Selective interpolation for SHORT gaps only
Only fills gaps up to 3 hours, only between real observations.
Long gaps stay NaN, honest.

In [10]:
def interp_short_gaps(s, limit=3):
    """Interpolate only interior gaps <= limit hours."""
    return s.interpolate(method="linear", limit=limit, limit_area="inside")

hourly["pm25_ug_m3"] = (
    hourly.groupby("station")["pm25_ug_m3"]
          .transform(interp_short_gaps)
)

# Recompute the missing flag AFTER interpolation so it reflects
# what is *still* missing in the model input.
hourly["pm25_missing"] = hourly["pm25_ug_m3"].isna().astype(int)

In [11]:
hourly.to_csv("../../Group2_ABT_hourly.csv", index=False)

## Transformation


### Temporal and Calendar Features
Encodes cyclical time components (hour, day of week, month) using sine/cosine transformations. Raw integer time values (e.g., hour=23 and hour=0) appear far apart to a model even though they are adjacent in reality; sinusoidal encoding preserves the natural periodicity of diurnal, weekly, and seasonal cycles. 

ref: https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0338392

In [12]:
hourly = pd.read_csv(ABT_DIR, parse_dates=["datetime_utc"])

dt = hourly["datetime_utc"].dt

# --- Cyclical encoding (keeps hour 23 adjacent to hour 0) ---
hourly["hour_sin"]  = np.sin(2 * np.pi * dt.hour / 24)
hourly["hour_cos"]  = np.cos(2 * np.pi * dt.hour / 24)

hourly["dow_sin"]   = np.sin(2 * np.pi * dt.dayofweek / 7)
hourly["dow_cos"]   = np.cos(2 * np.pi * dt.dayofweek / 7)

hourly["month_sin"] = np.sin(2 * np.pi * (dt.month - 1) / 12)
hourly["month_cos"] = np.cos(2 * np.pi * (dt.month - 1) / 12)

### Lag Features with Missing Indicators
Desc: Builds lagged PM2.5 values at multiple horizons (1, 3, 6, 12, 24, 48, 72 hours) and attaches a binary missingness flag to each. Lags capture autocorrelation and short-term persistence; the missing indicators let the model distinguish "sensor was down" from "value was zero," which is critical when stations have irregular coverage.

ref: https://arxiv.org/html/2607.07279v3

In [13]:
g = hourly.groupby("station")["pm25_ug_m3"]

for lag in [1, 3, 6, 12, 24, 48, 72]:
    col = f"pm25_lag{lag}"
    hourly[col] = g.transform(lambda s, l=lag: s.shift(l))
    hourly[f"{col}_missing"] = hourly[col].isna().astype(int)

## Derivative Features
Desc: Computes rate-of-change features that capture the velocity and acceleration of PM2.5 concentration. These detect transient pollution events (e.g., sudden AQI spikes) that raw values alone miss. Ratios compare the current value to recent baselines, revealing whether air quality is deteriorating relative to normal conditions.

ref: https://arxiv.org/html/2607.07279v3

In [14]:
g = hourly.groupby("station")["pm25_ug_m3"]

hourly["pm25_diff1"]  = g.transform(lambda s: s.diff(1))
hourly["pm25_diff24"] = g.transform(lambda s: s.diff(24))
hourly["pm25_accel"]  = hourly.groupby("station")["pm25_diff1"].transform(
    lambda s: s.diff(1)
)

# Ratios — compute the rolling means needed for these inline
roll24  = g.transform(lambda s: s.rolling(24,  min_periods=18).mean())
roll168 = g.transform(lambda s: s.rolling(168, min_periods=84).mean())

hourly["pm25_ratio_1h_24h"] = hourly["pm25_lag1"] / roll24
hourly["pm25_ratio_24h_7d"] = roll24 / roll168

## Rolling Statistics (supporting features)
Desc: Rolling mean, standard deviation, min, and max over multiple time windows. These summarize recent behavior of the PM2.5 series and are required inputs for the derivative ratio features above. min_periods prevents fake averages across long gaps.

ref: https://arxiv.org/html/2607.07279v3

In [15]:
g = hourly.groupby("station")["pm25_ug_m3"]

for w in [3, 6, 12, 24, 72, 168]:
    mp = max(2, w // 2)  # require at least half the window
    hourly[f"pm25_roll{w}_mean"] = g.transform(
        lambda s, w=w, mp=mp: s.rolling(w, min_periods=mp).mean()
    )
    hourly[f"pm25_roll{w}_std"] = g.transform(
        lambda s, w=w, mp=mp: s.rolling(w, min_periods=mp).std()
    )
    hourly[f"pm25_roll{w}_max"] = g.transform(
        lambda s, w=w, mp=mp: s.rolling(w, min_periods=mp).max()
    )
    hourly[f"pm25_roll{w}_min"] = g.transform(
        lambda s, w=w, mp=mp: s.rolling(w, min_periods=mp).min()
    )

## Station & City Context Features
Desc: Adds per-station and per-city summary statistics (mean, std, z-score, deviation). A global model pooling stations of unequal length needs context to know whether a given hourly value is "normal for this station" or "unusually high." The z-score and deviation features encode this directly.

ref: https://pmc.ncbi.nlm.nih.gov/articles/PMC8566180/

In [16]:
# Station-level aggregates
station_stats = (
    hourly.groupby("station")["pm25_ug_m3"]
          .agg(station_mean="mean",
               station_std="std")
)
hourly = hourly.merge(station_stats, on="station", how="left")

hourly["pm25_dev_from_station_mean"] = (
    hourly["pm25_ug_m3"] - hourly["station_mean"]
)
hourly["pm25_zscore"] = (
    (hourly["pm25_ug_m3"] - hourly["station_mean"]) / hourly["station_std"]
)

# City-level aggregates (mean of all stations in city per hour)
city_stats = (
    hourly.groupby(["city", "datetime_utc"])["pm25_ug_m3"]
          .mean()
          .rename("city_pm25_mean")
          .reset_index()
)
hourly = hourly.merge(city_stats, on=["city", "datetime_utc"], how="left")

hourly["pm25_dev_from_city"] = hourly["pm25_ug_m3"] - hourly["city_pm25_mean"]

##  Missingness Summary Features
Desc: Rolling averages of the missingness flag. High missingness often correlates with sensor outages, power failures during storms, or extreme events — informative signals the model can exploit. Complements the per-lag missing indicators from section 2.

ref: https://arxiv.org/html/2607.07279v3

In [17]:
g = hourly.groupby("station")["pm25_missing"]

hourly["missing_roll24"]  = g.transform(
    lambda s: s.rolling(24, min_periods=1).mean()
)
hourly["missing_roll168"] = g.transform(
    lambda s: s.rolling(168, min_periods=1).mean()
)

### THRESHOLD, LABELING

In [18]:
THRESHOLD = 35   # DENR AO

# STEP 1 — Ensure pm25_24h exists (backward-looking 24h mean)
if "pm25_24h" not in hourly.columns:
    g = hourly.groupby("station")["pm25_ug_m3"]
    hourly["pm25_24h"] = g.transform(
        lambda s: s.rolling(24, min_periods=18).mean()
    )


# STEP 2 — Ensure pm25_next24h exists (forward-looking 24h mean)
if "pm25_next24h" not in hourly.columns:
    g = hourly.groupby("station")["pm25_ug_m3"]
    hourly["pm25_next24h"] = g.transform(
        lambda s: s.rolling(24, min_periods=18).mean().shift(-24)
    )


# STEP 3 — Nowcast label (backward-looking, NaN-preserving)
alert_current = pd.Series(np.nan, index=hourly.index, dtype="float64")
mask_cur = hourly["pm25_24h"].notna()
alert_current.loc[mask_cur] = (
    hourly.loc[mask_cur, "pm25_24h"] > THRESHOLD
).astype(float)

hourly["alert_current_24h"] = alert_current.astype("Int64")


# STEP 4 — Preemptive label (forward-looking, NaN-preserving)
alert_next = pd.Series(np.nan, index=hourly.index, dtype="float64")
mask_next = hourly["pm25_next24h"].notna()
alert_next.loc[mask_next] = (
    hourly.loc[mask_next, "pm25_next24h"] > THRESHOLD
).astype(float)

hourly["alert_next24h"] = alert_next.astype("Int64")

print("=== pm25_24h / pm25_next24h ===")
print("pm25_24h NaN:     ", hourly["pm25_24h"].isna().sum())
print("pm25_next24h NaN: ", hourly["pm25_next24h"].isna().sum())

print("\n=== alert labels ===")
for col in ["alert_current_24h", "alert_next24h"]:
    print(f"\n{col}")
    print("  dtype:  ", hourly[col].dtype)
    print("  NaN:    ", hourly[col].isna().sum())
    print("  rate:   ", round(hourly[col].dropna().mean(), 4))
    print("  counts: ")
    print(hourly[col].value_counts(dropna=False))

=== pm25_24h / pm25_next24h ===
pm25_24h NaN:      57665
pm25_next24h NaN:  58199

=== alert labels ===

alert_current_24h
  dtype:   Int64
  NaN:     57665
  rate:    0.0773
  counts: 
alert_current_24h
0       395044
<NA>     57665
1        33072
Name: count, dtype: Int64

alert_next24h
  dtype:   Int64
  NaN:     58199
  rate:    0.0772
  counts: 
alert_next24h
0       394558
<NA>     58199
1        33024
Name: count, dtype: Int64


## Fixing leaky columns
station_mean, station_std, city_pm25_mean, pm25_zscore, pm25_dev_from_station_mean, pm25_dev_from_city were computed using all rows — including those that will become the test set. This leaks test information into training.

Fix: drop these columns from hourly, and recompute them inside your train/test pipeline.

In [19]:
leaky_cols = [
    "station_mean", "station_std",
    "pm25_dev_from_station_mean", "pm25_zscore",
    "city_pm25_mean", "pm25_dev_from_city",
]

hourly = hourly.drop(columns=leaky_cols)

In [20]:
hourly.to_csv("Group2_ABT_modeling.csv", index=False)

In [21]:
hourly = pd.read_csv("Group2_ABT_modeling.csv")

## MODELING

#### Step 1 — Imports and seed

In [22]:
import json
import joblib
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, average_precision_score,
    roc_auc_score, confusion_matrix,
)

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
print("Imports OK")

Imports OK


#### Step 2 — Load CSV and repair dtypes

In [23]:
# ---------- 1. Parse datetime back to UTC ----------
hourly["datetime_utc"] = pd.to_datetime(
    hourly["datetime_utc"], utc=True, errors="coerce"
)

# ---------- 2. Cast alert labels back to nullable Int64 ----------
for c in ["alert_current_24h", "alert_next24h"]:
    # First to float (in case it's object), then to Int64 (preserves NaN)
    hourly[c] = pd.to_numeric(hourly[c], errors="coerce").astype("Int64")

# ---------- 3. Verify ----------
print("datetime_utc :", hourly["datetime_utc"].dtype)
print("alert_current_24h:", hourly["alert_current_24h"].dtype,
      "| NaN:", hourly["alert_current_24h"].isna().sum())
print("alert_next24h:", hourly["alert_next24h"].dtype,
      "| NaN:", hourly["alert_next24h"].isna().sum())

print("\nValue counts for alert_current_24h:")
print(hourly["alert_current_24h"].value_counts(dropna=False))

print("\nValue counts for alert_next24h:")
print(hourly["alert_next24h"].value_counts(dropna=False))

datetime_utc : datetime64[ns, UTC]
alert_current_24h: Int64 | NaN: 57665
alert_next24h: Int64 | NaN: 58199

Value counts for alert_current_24h:
alert_current_24h
0       395044
<NA>     57665
1        33072
Name: count, dtype: Int64

Value counts for alert_next24h:
alert_next24h
0       394558
<NA>     58199
1        33024
Name: count, dtype: Int64


#### Step 3 — Time-based split

In [24]:
hourly = hourly.sort_values("datetime_utc").reset_index(drop=True)

cutoff = hourly["datetime_utc"].quantile(0.8)
train = hourly[hourly["datetime_utc"] <  cutoff].copy()
test  = hourly[hourly["datetime_utc"] >= cutoff].copy()

print("Train rows:", len(train),
      "| range:", train["datetime_utc"].min(), "→", train["datetime_utc"].max())
print("Test  rows:", len(test),
      "| range:", test["datetime_utc"].min(),  "→", test["datetime_utc"].max())

Train rows: 388622 | range: 2023-09-06 20:00:00+00:00 → 2026-03-12 18:00:00+00:00
Test  rows: 97159 | range: 2026-03-12 19:00:00+00:00 → 2026-09-04 07:00:00+00:00


#### Step 4 — Recompute context features on train only

In [25]:
# ---------- Station-level stats (fit on train) ----------
station_stats = (
    train.groupby("station")["pm25_ug_m3"]
         .agg(station_mean="mean", station_std="std")
         .reset_index()
)
train = train.merge(station_stats, on="station", how="left")
test  = test.merge(station_stats,  on="station", how="left")

for df in (train, test):
    df["station_std"] = df["station_std"].fillna(0)
    df["pm25_dev_from_station_mean"] = df["pm25_ug_m3"] - df["station_mean"]
    df["pm25_zscore"] = (
        (df["pm25_ug_m3"] - df["station_mean"]) / df["station_std"]
    ).fillna(0)

# ---------- City-level stats (fit on train) ----------
city_stats = (
    train.groupby(["city", "datetime_utc"])["pm25_ug_m3"]
         .mean().rename("city_pm25_mean").reset_index()
)
train = train.merge(city_stats, on=["city", "datetime_utc"], how="left")
test  = test.merge(city_stats,  on=["city", "datetime_utc"], how="left")

for df in (train, test):
    df["pm25_dev_from_city"] = df["pm25_ug_m3"] - df["city_pm25_mean"]

print("Context features added.")
print("train shape:", train.shape, "| test shape:", test.shape)

Context features added.
train shape: (388622, 67) | test shape: (97159, 67)


#### Step 5 — Define feature list

In [26]:
exclude_from_features = {
    # identifiers
    "datetime_utc", "city",
    # labels
    "alert_current_24h", "alert_next24h",
    # label source windows
    "pm25_24h", "pm25_24h_std", "pm25_next24h",
    # raw current value
    "pm25_ug_m3",
    # 24h-window features (residual leak)
    "pm25_roll24_mean", "pm25_roll24_std",
    "pm25_roll24_max",  "pm25_roll24_min",
    # 12h-window features
    "pm25_roll12_mean", "pm25_roll12_std",
    "pm25_roll12_max",  "pm25_roll12_min",
    # ratios on the 24h window
    "pm25_ratio_1h_24h", "pm25_ratio_24h_7d",
}

features = [c for c in train.columns if c not in exclude_from_features]

features = [c for c in train.columns if c not in exclude_from_features]
print("n features:", len(features))
print(features)

# Persist for reproducibility
with open("feature_list.json", "w") as f:
    json.dump(features, f, indent=2)

n features: 50
['station', 'density', 'pm25_missing', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'pm25_lag1', 'pm25_lag1_missing', 'pm25_lag3', 'pm25_lag3_missing', 'pm25_lag6', 'pm25_lag6_missing', 'pm25_lag12', 'pm25_lag12_missing', 'pm25_lag24', 'pm25_lag24_missing', 'pm25_lag48', 'pm25_lag48_missing', 'pm25_lag72', 'pm25_lag72_missing', 'pm25_diff1', 'pm25_diff24', 'pm25_accel', 'pm25_roll3_mean', 'pm25_roll3_std', 'pm25_roll3_max', 'pm25_roll3_min', 'pm25_roll6_mean', 'pm25_roll6_std', 'pm25_roll6_max', 'pm25_roll6_min', 'pm25_roll72_mean', 'pm25_roll72_std', 'pm25_roll72_max', 'pm25_roll72_min', 'pm25_roll168_mean', 'pm25_roll168_std', 'pm25_roll168_max', 'pm25_roll168_min', 'missing_roll24', 'missing_roll168', 'station_mean', 'station_std', 'pm25_dev_from_station_mean', 'pm25_zscore', 'city_pm25_mean', 'pm25_dev_from_city']


#### Step 6 — Drop NaN targets per model

In [27]:
TARGET_REG = "pm25_next24h"   # ← was "pm25_24h"
TARGET_NOW = "alert_current_24h"
TARGET_NXT = "alert_next24h"

# Regression — target = pm25_next24h (forward-looking)
train_reg = train.dropna(subset=["pm25_next24h"]).copy()
test_reg  = test.dropna(subset=["pm25_next24h"]).copy()

# Nowcast classifier — unchanged (backward target is its nature)
train_now = train.dropna(subset=["alert_current_24h"]).copy()
test_now  = test.dropna(subset=["alert_current_24h"]).copy()

# Preemptive classifier — unchanged
train_nxt = train.dropna(subset=["alert_next24h"]).copy()
test_nxt  = test.dropna(subset=["alert_next24h"]).copy()

print("Regression  train/test:", len(train_reg), len(test_reg))
print("Nowcast     train/test:", len(train_now), len(test_now))
print("Preemptive  train/test:", len(train_nxt), len(test_nxt))

Regression  train/test: 345457 82125
Nowcast     train/test: 344979 83137
Preemptive  train/test: 345457 82125


In [28]:
# ---- Cast station to category once, for all downstream models ----
cats = sorted(train_reg["station"].astype(str).unique())

train_reg["station"] = pd.Categorical(train_reg["station"].astype(str), categories=cats)
test_reg["station"]  = pd.Categorical(test_reg["station"].astype(str),  categories=cats)

train_now["station"] = pd.Categorical(train_now["station"].astype(str), categories=cats)
test_now["station"]  = pd.Categorical(test_now["station"].astype(str),  categories=cats)

train_nxt["station"] = pd.Categorical(train_nxt["station"].astype(str), categories=cats)
test_nxt["station"]  = pd.Categorical(test_nxt["station"].astype(str),  categories=cats)

print("station dtype:", train_reg["station"].dtype)
print("n categories: ", len(cats))

station dtype: category
n categories:  71


##### Step 6a — Ridge tuning

In [29]:
# ---- Rebuild one-hot features (as before) ----
train_ridge = pd.get_dummies(
    train_reg[features], columns=["station"], prefix="st", dummy_na=False
)
test_ridge = pd.get_dummies(
    test_reg[features],  columns=["station"], prefix="st", dummy_na=False
)
train_ridge, test_ridge = train_ridge.align(
    test_ridge, join="left", axis=1, fill_value=0
)

# ---- Pipeline ----
ridge_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
    ("model",  Ridge(random_state=SEED)),
])

# ---- Time-series cross-validation (no shuffling) ----
tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "model__alpha": [0.01, 0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000],
}

grid = GridSearchCV(
    ridge_pipe,
    param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)
grid.fit(train_ridge, train_reg[TARGET_REG])

print("Best alpha:", grid.best_params_)
print("Best CV RMSE:", -grid.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best alpha: {'model__alpha': 1000}
Best CV RMSE: 6.615164439794173


##### Step 6b — LightGBM tuning (Optuna)

In [30]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---- Define inner validation slice ----
val_cutoff = train_reg["datetime_utc"].quantile(0.8)
train_fit  = train_reg[train_reg["datetime_utc"] <  val_cutoff].copy()
train_val  = train_reg[train_reg["datetime_utc"] >= val_cutoff].copy()

# ---- Cast station to category HERE, on the slices actually used ----
cats = sorted(train_reg["station"].astype(str).unique())
for df in (train_fit, train_val):
    df["station"] = pd.Categorical(df["station"].astype(str), categories=cats)

# ---- Sanity check ----
bad = [c for c in features if train_fit[c].dtype == "object"]
print("fit rows:", len(train_fit), "| val rows:", len(train_val))
print("object-dtype features:", bad)   # must be []


# ---- Objective ----
def objective(trial):
    params = {
        "n_estimators":      5000,
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 15, 127),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        "random_state":      SEED,
        "n_jobs":            -1,
        "verbosity":         -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(
        train_fit[features], train_fit[TARGET_REG],
        eval_set=[(train_val[features], train_val[TARGET_REG])],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    pred = model.predict(train_val[features])
    return np.sqrt(mean_squared_error(train_val[TARGET_REG], pred))

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("Best val RMSE:", study.best_value)

best_lgbm_params = study.best_params

fit rows: 276347 | val rows: 69110
object-dtype features: []


  0%|          | 0/30 [00:00<?, ?it/s]

c:\Users\Lenovo Thinkpad T460\OneDrive\Anaconda\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\Lenovo Thinkpad T460\OneDrive\Anaconda\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\Lenovo Thinkpad T460\OneDrive\Anaconda\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\Lenovo Thinkpad T460\OneDrive\Anaconda\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_

Best params:
  learning_rate: 0.02390553492784971
  num_leaves: 22
  min_child_samples: 136
  subsample: 0.6281983445485977
  colsample_bytree: 0.6335000638576233
  reg_alpha: 3.763063766267415
  reg_lambda: 7.092234763326141
Best val RMSE: 7.029799158262282


#### Step 7 — Global LightGBM (regression, champion)

In [31]:
# ---- Cast station ----
cats = sorted(train_reg["station"].astype(str).unique())
train_reg["station"] = pd.Categorical(train_reg["station"].astype(str), categories=cats)
test_reg["station"]  = pd.Categorical(test_reg["station"].astype(str),  categories=cats)

# ---- Inner validation slice ----
val_cutoff = train_reg["datetime_utc"].quantile(0.8)
train_fit  = train_reg[train_reg["datetime_utc"] <  val_cutoff]
train_val  = train_reg[train_reg["datetime_utc"] >= val_cutoff]

# ---- Final model with tuned params ----
model_lgbm = lgb.LGBMRegressor(
    n_estimators=5000,
    **best_lgbm_params,          # ← tuned hyperparameters
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

model_lgbm.fit(
    train_fit[features], train_fit[TARGET_REG],
    eval_set=[(train_val[features], train_val[TARGET_REG])],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=200),
    ],
)
print("Best iteration:", model_lgbm.best_iteration_)

# ---- Evaluate on held-out test ----
pred = model_lgbm.predict(test_reg[features])
y    = test_reg[TARGET_REG].to_numpy()

rmse = np.sqrt(mean_squared_error(y, pred))
mae  = mean_absolute_error(y, pred)
r2   = r2_score(y, pred)

print("\n=== Global LightGBM (tuned) ===")
print(f"RMSE : {rmse:.3f}")
print(f"MAE  : {mae:.3f}")
print(f"R²   : {r2:.4f}")

# ---- Fair baseline comparison ----
mask = test_reg["pm25_lag24"].notna()
y_subset    = test_reg.loc[mask, TARGET_REG].to_numpy()
pred_subset = pred[mask]
base_subset = test_reg.loc[mask, "pm25_lag24"].to_numpy()

rmse_model_sub = np.sqrt(mean_squared_error(y_subset, pred_subset))
rmse_base_sub  = np.sqrt(mean_squared_error(y_subset, base_subset))

print(f"\nFair comparison on {mask.sum():,} rows:")
print(f"Model    → RMSE: {rmse_model_sub:.3f}")
print(f"Baseline → RMSE: {rmse_base_sub:.3f}")
print(f"Improvement: {(1 - rmse_model_sub / rmse_base_sub) * 100:.2f}%")

# ---- Feature importance ----
importance = (
    pd.Series(model_lgbm.feature_importances_, index=features)
      .sort_values(ascending=False)
)
print("\nTop 20 features:")
print(importance.head(20))

joblib.dump(model_lgbm, "global_lgbm_pm25_tuned.pkl")
print("\nSaved: global_lgbm_pm25_tuned.pkl")

c:\Users\Lenovo Thinkpad T460\OneDrive\Anaconda\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[200]	valid_0's rmse: 7.05725	valid_0's l2: 49.8048
Best iteration: 289

=== Global LightGBM (tuned) ===
RMSE : 9.699
MAE  : 6.499
R²   : 0.4370

Fair comparison on 79,124 rows:
Model    → RMSE: 9.750
Baseline → RMSE: 16.101
Improvement: 39.45%

Top 20 features:
station              572
pm25_roll168_min     346
pm25_roll72_min      330
month_cos            290
pm25_roll72_mean     269
city_pm25_mean       263
month_sin            261
pm25_roll72_max      258
pm25_roll168_mean    244
missing_roll168      241
pm25_roll168_max     226
pm25_lag12           224
dow_cos              219
dow_sin              215
station_mean         202
pm25_roll168_std     173
pm25_roll6_min       168
missing_roll24       140
pm25_roll72_std      132
pm25_diff1           130
dtype: int32

Saved: global_lgbm_pm25_tuned.pkl


#### Step 8 — Ridge regression (challenger)

In [32]:
best_alpha = grid.best_params_["model__alpha"]
print("Using alpha =", best_alpha)

model_ridge = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
    ("model",  Ridge(alpha=best_alpha, random_state=SEED)),
])

model_ridge.fit(train_ridge, train_reg[TARGET_REG])

# ---- Evaluate ----
pred_r = model_ridge.predict(test_ridge)
y_r    = test_reg[TARGET_REG].to_numpy()

rmse_r = np.sqrt(mean_squared_error(y_r, pred_r))
mae_r  = mean_absolute_error(y_r, pred_r)
r2_r   = r2_score(y_r, pred_r)

print("\n=== Ridge Regression (tuned) ===")
print(f"RMSE : {rmse_r:.3f}")
print(f"MAE  : {mae_r:.3f}")
print(f"R²   : {r2_r:.4f}")

joblib.dump(model_ridge, "ridge_pm25_tuned.pkl")
print("Saved: ridge_pm25_tuned.pkl")

Using alpha = 1000

=== Ridge Regression (tuned) ===
RMSE : 9.282
MAE  : 6.294
R²   : 0.4844
Saved: ridge_pm25_tuned.pkl


#### Step 9 — LightGBM Classifier (Nowcast — alert_current_24h)

In [33]:
# ---- Cast station ----
cats_now = sorted(train_now["station"].astype(str).unique())
train_now["station"] = pd.Categorical(train_now["station"].astype(str), categories=cats_now)
test_now["station"]  = pd.Categorical(test_now["station"].astype(str),  categories=cats_now)

# ---- Inner validation slice ----
val_cutoff_now = train_now["datetime_utc"].quantile(0.8)
train_now_fit  = train_now[train_now["datetime_utc"] <  val_cutoff_now]
train_now_val  = train_now[train_now["datetime_utc"] >= val_cutoff_now]

# ---- Class balance ----
rate_now = train_now_fit[TARGET_NOW].mean()
spw_now  = (1 - rate_now) / rate_now
print(f"Nowcast positive rate: {rate_now:.4f} | scale_pos_weight = {spw_now:.2f}")

# ---- Model (tuned) ----
clf_now = lgb.LGBMClassifier(
    n_estimators=5000,
    **best_lgbm_params,           # ← tuned hyperparameters from Step 6b
    scale_pos_weight=spw_now,     # derived, not tuned
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

clf_now.fit(
    train_now_fit[features], train_now_fit[TARGET_NOW].astype(int),
    eval_set=[(train_now_val[features], train_now_val[TARGET_NOW].astype(int))],
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=200),
    ],
)
print("Best iteration:", clf_now.best_iteration_)

# ---- Evaluate ----
proba = clf_now.predict_proba(test_now[features])[:, 1]
pred  = (proba >= 0.5).astype(int)
y_now = test_now[TARGET_NOW].astype(int).to_numpy()

print("\n=== LightGBM Classifier (Nowcast, tuned) ===")
print(classification_report(y_now, pred, digits=3))
print("PR-AUC:", round(average_precision_score(y_now, proba), 4))
print("ROC-AUC:", round(roc_auc_score(y_now, proba), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_now, pred))

joblib.dump(clf_now, "clf_nowcast_tuned.pkl")
print("Saved: clf_nowcast_tuned.pkl")

Nowcast positive rate: 0.0646 | scale_pos_weight = 14.47


c:\Users\Lenovo Thinkpad T460\OneDrive\Anaconda\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Best iteration: 81

=== LightGBM Classifier (Nowcast, tuned) ===
              precision    recall  f1-score   support

           0      0.991     0.913     0.950     70908
           1      0.653     0.954     0.775     12229

    accuracy                          0.919     83137
   macro avg      0.822     0.933     0.863     83137
weighted avg      0.942     0.919     0.925     83137

PR-AUC: 0.9155
ROC-AUC: 0.9829

Confusion matrix:
[[64709  6199]
 [  568 11661]]
Saved: clf_nowcast_tuned.pkl


#### Step 10 — LightGBM Classifier (Preemptive — alert_next24h)

In [34]:
# ---- Cast station ----
cats_nxt = sorted(train_nxt["station"].astype(str).unique())
train_nxt["station"] = pd.Categorical(train_nxt["station"].astype(str), categories=cats_nxt)
test_nxt["station"]  = pd.Categorical(test_nxt["station"].astype(str),  categories=cats_nxt)

# ---- Inner validation slice ----
val_cutoff_nxt = train_nxt["datetime_utc"].quantile(0.8)
train_nxt_fit  = train_nxt[train_nxt["datetime_utc"] <  val_cutoff_nxt]
train_nxt_val  = train_nxt[train_nxt["datetime_utc"] >= val_cutoff_nxt]

# ---- Class balance ----
rate_nxt = train_nxt_fit[TARGET_NXT].mean()
spw_nxt  = (1 - rate_nxt) / rate_nxt
print(f"Preemptive positive rate: {rate_nxt:.4f} | scale_pos_weight = {spw_nxt:.2f}")

# ---- Model (tuned) ----
clf_nxt = lgb.LGBMClassifier(
    n_estimators=5000,
    **best_lgbm_params,           # ← tuned hyperparameters from Step 6b
    scale_pos_weight=spw_nxt,     # derived, not tuned
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

clf_nxt.fit(
    train_nxt_fit[features], train_nxt_fit[TARGET_NXT].astype(int),
    eval_set=[(train_nxt_val[features], train_nxt_val[TARGET_NXT].astype(int))],
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=200),
    ],
)
print("Best iteration:", clf_nxt.best_iteration_)

# ---- Evaluate ----
proba_n = clf_nxt.predict_proba(test_nxt[features])[:, 1]
pred_n  = (proba_n >= 0.5).astype(int)
y_nxt   = test_nxt[TARGET_NXT].astype(int).to_numpy()

print("\n=== LightGBM Classifier (Preemptive, tuned) ===")
print(classification_report(y_nxt, pred_n, digits=3))
print("PR-AUC:", round(average_precision_score(y_nxt, proba_n), 4))
print("ROC-AUC:", round(roc_auc_score(y_nxt, proba_n), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_nxt, pred_n))

joblib.dump(clf_nxt, "clf_preemptive_tuned.pkl")
print("Saved: clf_preemptive_tuned.pkl")

Preemptive positive rate: 0.0645 | scale_pos_weight = 14.50


c:\Users\Lenovo Thinkpad T460\OneDrive\Anaconda\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Best iteration: 7

=== LightGBM Classifier (Preemptive, tuned) ===
              precision    recall  f1-score   support

           0      0.851     1.000     0.920     69906
           1      0.000     0.000     0.000     12219

    accuracy                          0.851     82125
   macro avg      0.426     0.500     0.460     82125
weighted avg      0.725     0.851     0.783     82125

PR-AUC: 0.4179
ROC-AUC: 0.804

Confusion matrix:
[[69906     0]
 [12219     0]]
Saved: clf_preemptive_tuned.pkl


c:\Users\Lenovo Thinkpad T460\OneDrive\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo Thinkpad T460\OneDrive\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo Thinkpad T460\OneDrive\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{me

#### Step 11 — Compare all models

In [35]:
results = pd.DataFrame({
    "Model":  ["Global LightGBM", "Ridge", "Nowcast Clf", "Preemptive Clf"],
    "Task":   ["Regression", "Regression", "Classification", "Classification"],
    "RMSE":   [rmse, rmse_r, np.nan, np.nan],
    "MAE":    [mae, mae_r, np.nan, np.nan],
    "R²":     [r2, r2_r, np.nan, np.nan],
    "PR-AUC": [np.nan, np.nan,
               average_precision_score(y_now, proba),
               average_precision_score(y_nxt, proba_n)],
})
print(results.round(4).to_string(index=False))

          Model           Task   RMSE    MAE     R²  PR-AUC
Global LightGBM     Regression 9.6989 6.4995 0.4370     NaN
          Ridge     Regression 9.2818 6.2938 0.4844     NaN
    Nowcast Clf Classification    NaN    NaN    NaN  0.9155
 Preemptive Clf Classification    NaN    NaN    NaN  0.4179


FINDINGS:

* Ridge regression beat Global LightGBM
* Nowcasting is good, Forecasting is hard which is expected since the 
* Published literatures with only regressive features tops out at R^2 0.4-0.6. This findings aligns with ours (0.44–0.48).
* Meteorological features are a must: wind, humidity, temperature, boundary layer, precipitation are used in almost all study. However, the current data none any of these which might be the reason.

#### Conclusion:
A leak-free, time-validated pipeline shows that 24h-ahead PM2.5 is moderately predictable (R² ~0.48) with a dominantly linear signal, that nowcasting exceedances is easy while preemptive prediction remains hard but feasible (PR-AUC ~0.42), and that meteorological features are the most promising next lever to lift forecasting performance.

### Recommendation in using this model output:
* Deploy a current exeedances as it has high result (~92).
* For the next 24hr alert prediction, use only as internal early-warning, not yet for automatic public alert
* For predicting PM 2.5 magnitudes, the regression models predicted with 9-10 units error and an r^2 of ~.48. This should be use only for severity grading, not a precise numeric forecasts.